# Sun Diagnostic Visualization

Split from `diagnostic_sun.ipynb`: shared setup cells followed by data-quality and inspection plots.

In [ ]:
import sys
sys.path.insert(0, '.')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.coordinates import get_sun
from astropy.time import Time

from utils.plotter import (
    plot_capture_timeline_and_gaps,
    plot_channel_time_series,
    plot_example_spectrum,
    plot_unwrapped_phase_vs_ha_time,
    plot_visibility_waterfall,
    plot_waterfall_amplitude,
)


In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────
DATA_DIR = Path('../../data/lab03/sun_calibration')

LO1_HZ = 8750e6
LO2_HZ = 1540e6
IF1_BPF_CENTER_HZ = 1700e6
IF1_BPF_HALF_BW_HZ = 35e6
PLOT_BAND_GHZ = (
    (LO1_HZ + IF1_BPF_CENTER_HZ - IF1_BPF_HALF_BW_HZ) / 1e9,
    (LO1_HZ + IF1_BPF_CENTER_HZ + IF1_BPF_HALF_BW_HZ) / 1e9,
)

paths = sorted(DATA_DIR.glob('*.npz'))
files = [np.load(p) for p in paths]

f0 = files[0]
F_S_HZ = float(f0['f_s_hz']) if 'f_s_hz' in f0 else 500e6
FILE_F_RF0_HZ = float(f0['f_rf0_hz']) if 'f_rf0_hz' in f0 else float('nan')
F_RF0_HZ = LO1_HZ + LO2_HZ
N_CH = int(f0['corr'].shape[0])
N_FFT = int(f0['n_fft']) if 'n_fft' in f0 else 2048
DF_HZ = F_S_HZ / N_FFT
F_SKY_GHZ = (F_RF0_HZ + np.arange(N_CH) * DF_HZ) / 1e9

print(f'Loaded {len(files)} captures  |  '
      f'{F_SKY_GHZ[0]:.4f} -- {F_SKY_GHZ[-1]:.4f} GHz  |  '
      f'{DF_HZ/1e3:.1f} kHz/ch')
print(f'Analysis band : {PLOT_BAND_GHZ[0]:.3f} -- {PLOT_BAND_GHZ[1]:.3f} GHz')
if np.isfinite(FILE_F_RF0_HZ) and not np.isclose(FILE_F_RF0_HZ, F_RF0_HZ):
    print(
        f'Ignoring file metadata f_rf0_hz={FILE_F_RF0_HZ/1e9:.4f} GHz; '
        f'using LO-derived {F_RF0_HZ/1e9:.4f} GHz'
    )

# ── Mask ──────────────────────────────────────────────────────────────────
BAD_CHANNELS = [0, 256, 512, 768]   # DC/LO leak + FPGA harmonics (N_FFT/8)

all_amp                  = np.array([np.abs(f['corr']) for f in files]).astype(float)
all_amp[:, BAD_CHANNELS] = np.nan

print(f'Masked : {BAD_CHANNELS}  '
      f'({[f"{F_SKY_GHZ[k]:.4f} GHz" for k in BAD_CHANNELS]})')

# ── Normalise ─────────────────────────────────────────────────────────────
AMP_PEAK = np.nanmax(all_amp)
print(f'Peak   : {AMP_PEAK:.4g}')


In [ ]:
# ---------------------------------------------------------------------------
# Compute hour angle for each capture (sorted chronologically)
# The Sun's RA changes ~1°/day, so it must be recomputed per capture.
# ---------------------------------------------------------------------------

NCH_LON_DEG = -122.2573   # NCH site longitude (degrees east)

def _mid_unix(f):
    """Midpoint unix timestamp; falls back to legacy 'unix_time' key."""
    if 'unix_time_start' in f:
        return (float(f['unix_time_start']) + float(f['unix_time_end'])) / 2
    return float(f['unix_time'])

def _lst_deg(unix_t):
    jd = unix_t / 86400.0 + 2440587.5
    T  = (jd - 2451545.0) / 36525.0
    g  = (280.46061837 + 360.98564736629 * (jd - 2451545.0)
          + T**2 * 0.000387933 - T**3 / 38710000.0)
    return (g + NCH_LON_DEG) % 360.0

def _sun_ra_deg(unix_t):
    """Sun's J2000 RA in degrees at the given unix time."""
    return get_sun(Time(unix_t, format='unix')).ra.deg

# Sort files by mid-capture unix time
unix_mid  = np.array([_mid_unix(f) for f in files])
order     = np.argsort(unix_mid)
unix_sort = unix_mid[order]
files_s   = [files[j]  for j in order]
paths_s   = [paths[j]  for j in order]
N_cap     = len(files_s)

lst_arr     = np.array([_lst_deg(t)    for t in unix_sort])
sun_ra_arr  = np.array([_sun_ra_deg(t) for t in unix_sort])
ha_deg      = (lst_arr - sun_ra_arr) % 360.0
ha_deg[ha_deg > 180.0] -= 360.0

# ---------------------------------------------------------------------------
# Split into 2 chips at the largest inter-capture gap
# ---------------------------------------------------------------------------
_t_end_arr   = np.array([float(f['unix_time_end'])   for f in files_s])
_t_start_arr = np.array([float(f['unix_time_start']) for f in files_s])
_gaps        = _t_start_arr[1:] - _t_end_arr[:-1]
_split       = int(np.argmax(_gaps)) + 1          # first index of chip 1

_slices = [slice(0, _split), slice(_split, N_cap)]

# Per-chip views of every per-capture array
files_chips    = [[files_s[i]   for i in range(s.start, s.stop)] for s in _slices]
unix_chips     = [unix_sort[s]  for s in _slices]
ha_chips       = [ha_deg[s]     for s in _slices]
N_caps         = [s.stop - s.start for s in _slices]

print(f'Gap at index {_split-1}→{_split}: {_gaps[_split-1]:.1f} s  '
      f'({_gaps[_split-1]/60:.1f} min)')
print(f'Chip 0: {N_caps[0]} captures,  HA {ha_chips[0].min():.2f}° → {ha_chips[0].max():.2f}°')
print(f'Chip 1: {N_caps[1]} captures,  HA {ha_chips[1].min():.2f}° → {ha_chips[1].max():.2f}°')

# Peak channel within PLOT_BAND_GHZ (excluding artifact channels)
_ch_lo  = int(np.searchsorted(F_SKY_GHZ, PLOT_BAND_GHZ[0]))
_ch_hi  = int(np.searchsorted(F_SKY_GHZ, PLOT_BAND_GHZ[1]))
k_peak  = _ch_lo + int(np.nanargmax(np.nanmean(all_amp, axis=0)[_ch_lo:_ch_hi]))
F_PEAK_GHZ = F_SKY_GHZ[k_peak]

vis_peak   = np.array([f['corr'][k_peak] for f in files_s])
amp_peak   = np.abs(vis_peak) / AMP_PEAK
phase_peak = np.rad2deg(np.angle(vis_peak))

print(f'Sun RA   : {sun_ra_arr.min():.4f} -- {sun_ra_arr.max():.4f} deg  '
      f'(span {sun_ra_arr.max()-sun_ra_arr.min():.4f} deg)')
print(f'HA range : {ha_deg.min():.2f}° -> {ha_deg.max():.2f}°  '
      f'({ha_deg.max() - ha_deg.min():.2f}° span)')
print(f'Peak channel : k={k_peak},  f_sky={F_PEAK_GHZ:.4f} GHz')
print(f'Amp  (norm)  : mean={amp_peak.mean():.3f},  std={amp_peak.std():.3f}')
print(f'Phase        : mean={phase_peak.mean():.1f}°,  std={phase_peak.std():.1f}°')

In [ ]:
# ---------------------------------------------------------------------------
# DC offset correction per chip — avoids cross-chip mean contamination.
# Each chip's DC is estimated from within that chip only.
# corr_dc_chips[c] : (N_caps[c], N_CH)  DC-corrected complex visibility
# corr_dc           : (N_cap, N_CH)     full sequence (for non-gap-sensitive cells)
# ---------------------------------------------------------------------------

corr_dc_chips = []
for fchip in files_chips:
    mat = np.array([f['corr'].astype(complex) for f in fchip])   # (N_c, N_CH)
    mat[:, BAD_CHANNELS] = np.nan
    re_mean_c = np.nanmean(mat.real, axis=0)
    corr_dc_chips.append(mat - re_mean_c[np.newaxis, :])

corr_dc = np.vstack(corr_dc_chips)   # (N_cap, N_CH) — full sequence

In [ ]:
# ---------------------------------------------------------------------------
# Example spectrum — median capture
# ---------------------------------------------------------------------------

i_med    = len(files_s) // 2
f_ex     = files_s[i_med]
amp_ex   = np.abs(f_ex['corr']).astype(float) / AMP_PEAK
amp_ex[BAD_CHANNELS] = np.nan

integ_s  = float(f_ex['unix_time_end']) - float(f_ex['unix_time_start'])
n_acc_ex = int(f_ex['n_acc'])
alt_ex   = float(f_ex['alt_deg']) if 'alt_deg' in f_ex else float('nan')
ha_ex    = ha_deg[i_med]

fig, ax = plot_example_spectrum(
    f_sky_ghz=F_SKY_GHZ,
    amp_norm=amp_ex,
    plot_band_ghz=PLOT_BAND_GHZ,
    peak_freq_ghz=F_PEAK_GHZ,
    f_rf0_hz=F_RF0_HZ,
    df_hz=DF_HZ,
    capture_index=i_med,
    capture_count=len(files_s),
    integration_s=integ_s,
    n_acc=n_acc_ex,
    ha_deg=ha_ex,
    alt_deg=alt_ex,
)
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# Waterfall (spectrogram): f_sky vs hour angle
# ---------------------------------------------------------------------------

amp_matrix = np.array([np.abs(f['corr']) / AMP_PEAK for f in files_s]).astype(float)
amp_matrix[:, BAD_CHANNELS] = np.nan

fig, axes = plot_waterfall_amplitude(
    f_sky_ghz=F_SKY_GHZ,
    ha_deg=ha_deg,
    amp_matrix=amp_matrix,
    plot_band_ghz=PLOT_BAND_GHZ,
    f_rf0_hz=F_RF0_HZ,
    df_hz=DF_HZ,
    title=rf'Sun --- {len(files_s)}\,captures',
    cbar_label=r'$|V_{12}| / |V_{12}|_{\rm peak}$',
)
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# Waterfalls: arg(V_12),  Re(V_12),  Im(V_12)  — f_sky vs hour angle
# ---------------------------------------------------------------------------

_wf_configs = [
    ('arg',  r'$\arg(V_{12})$ [deg]',  'twilight', -180, 180),
    ('re',   r'$\mathrm{Re}(V_{12})$', 'RdBu_r',   None, None),
    ('im',   r'$\mathrm{Im}(V_{12})$', 'RdBu_r',   None, None),
]

for kind, cbar_label, cmap_name, vmin_fixed, vmax_fixed in _wf_configs:
    rows = []
    for f in files_s:
        corr = f['corr'].astype(complex)
        if kind == 'arg':
            row = np.rad2deg(np.angle(corr))
        elif kind == 're':
            row = corr.real
        else:
            row = corr.imag
        row = row.copy()
        row[BAD_CHANNELS] = np.nan
        rows.append(row)
    mat = np.array(rows)

    if vmin_fixed is None:
        vabs = np.nanpercentile(np.abs(mat), 99)
        vmin, vmax = -vabs, vabs
    else:
        vmin, vmax = vmin_fixed, vmax_fixed

    fig, axes = plot_visibility_waterfall(
        f_sky_ghz=F_SKY_GHZ,
        ha_deg=ha_deg,
        quantity_matrix=mat,
        plot_band_ghz=PLOT_BAND_GHZ,
        f_rf0_hz=F_RF0_HZ,
        df_hz=DF_HZ,
        title=rf'Sun --- {len(files_s)}\,captures',
        cbar_label=cbar_label,
        cmap_name=cmap_name,
        vmin=vmin,
        vmax=vmax,
    )
    plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# Integration time coverage
# ---------------------------------------------------------------------------

t_start = np.array([float(f['unix_time_start']) for f in files_s])
t_end   = np.array([float(f['unix_time_end'])   for f in files_s])
n_acc   = np.array([int(f['n_acc'])             for f in files_s])
durs    = t_end - t_start
gaps    = t_start[1:] - t_end[:-1]
t0      = t_start[0]

total_wall = t_end[-1] - t_start[0]
total_sky  = durs.sum()
duty       = total_sky / total_wall * 100

print(f'Captures        : {len(files_s)}')
print(f'Wall-clock span : {total_wall:.1f} s  ({total_wall/60:.1f} min)')
print(f'On-sky total    : {total_sky:.1f} s  ({total_sky/60:.1f} min)')
print(f'Duty cycle      : {duty:.1f}%')
print(f'n_acc per cap   : min={n_acc.min()}  mean={n_acc.mean():.1f}  max={n_acc.max()}')
print(f'Gap (s)         : min={gaps.min():.2f}  median={np.median(gaps):.2f}  mean={gaps.mean():.2f}  max={gaps.max():.2f}')

fig, axes = plot_capture_timeline_and_gaps(
    capture_start_s=t_start - t0,
    capture_end_s=t_end - t0,
    capture_count=len(files_s),
    duty_cycle_pct=duty,
    gap_s=gaps,
)
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# DC-corrected waterfalls: amp, arg, Re, Im  — f_sky vs hour angle
# ---------------------------------------------------------------------------

amp_dc = np.abs(corr_dc)
amp_dc_peak = np.nanmax(amp_dc)

_wf_configs_dc = [
    (
        'amp',
        r'$|V_{12} - \langle\mathrm{Re}\rangle|\,/\,|V_{12} - \langle\mathrm{Re}\rangle|_{\rm peak}$',
        'viridis',
        0,
        1,
    ),
    ('arg', r'$\arg(V_{12} - \langle\mathrm{Re}\rangle)$ [deg]', 'twilight', -180, 180),
    ('re', r'$\mathrm{Re}(V_{12}) - \langle\mathrm{Re}\rangle$', 'RdBu_r', None, None),
    ('im', r'$\mathrm{Im}(V_{12})$', 'RdBu_r', None, None),
]

for kind, cbar_label, cmap_name, vmin_fixed, vmax_fixed in _wf_configs_dc:
    if kind == 'amp':
        mat = amp_dc / amp_dc_peak
    elif kind == 'arg':
        mat = np.rad2deg(np.angle(corr_dc))
    elif kind == 're':
        mat = corr_dc.real
    else:
        mat = corr_dc.imag

    if vmin_fixed is None:
        vabs = np.nanpercentile(np.abs(mat), 99)
        vmin, vmax = -vabs, vabs
    else:
        vmin, vmax = vmin_fixed, vmax_fixed

    if kind == 'amp':
        fig, axes = plot_waterfall_amplitude(
            f_sky_ghz=F_SKY_GHZ,
            ha_deg=ha_deg,
            amp_matrix=mat,
            plot_band_ghz=PLOT_BAND_GHZ,
            f_rf0_hz=F_RF0_HZ,
            df_hz=DF_HZ,
            title=rf'Sun (DC corrected) --- {len(files_s)}\,captures',
            cbar_label=cbar_label,
        )
    else:
        fig, axes = plot_visibility_waterfall(
            f_sky_ghz=F_SKY_GHZ,
            ha_deg=ha_deg,
            quantity_matrix=mat,
            plot_band_ghz=PLOT_BAND_GHZ,
            f_rf0_hz=F_RF0_HZ,
            df_hz=DF_HZ,
            title=rf'Sun (DC corrected) --- {len(files_s)}\,captures',
            cbar_label=cbar_label,
            cmap_name=cmap_name,
            vmin=vmin,
            vmax=vmax,
        )
    plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# Per-channel time series — amplitude, arg, Re, Im vs hour angle
# Both chips overlaid on each panel.
# ---------------------------------------------------------------------------

K_CH = 600
_f_ch = F_SKY_GHZ[K_CH]

# Global amplitude mean across both chips for a common normalisation
_amp_global_mean = np.nanmean(
    np.concatenate([np.abs(dc_c[:, K_CH]) for dc_c in corr_dc_chips])
)

_amp_norm_chips = []
_phase_deg_chips = []
_real_chips = []
_imag_chips = []

for dc_c in corr_dc_chips:
    _ch_vis = dc_c[:, K_CH]
    _amp_norm_chips.append(np.abs(_ch_vis) / _amp_global_mean)
    _phase_deg_chips.append(np.rad2deg(np.angle(_ch_vis)))
    _real_chips.append(_ch_vis.real)
    _imag_chips.append(_ch_vis.imag)

fig, axes = plot_channel_time_series(
    ha_chips=ha_chips,
    amp_norm_chips=_amp_norm_chips,
    phase_deg_chips=_phase_deg_chips,
    real_chips=_real_chips,
    imag_chips=_imag_chips,
    channel_index=K_CH,
    channel_freq_ghz=_f_ch,
    ha_limits_deg=(ha_deg.min(), ha_deg.max()),
)
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# Unwrapped phase vs HA-time — channel K_CH
# Unwrap per chip to avoid phase jump at the gap.
# ---------------------------------------------------------------------------

_omega_sid_deg_s = (2 * np.pi / 86164.1) * (180 / np.pi)  # deg/s
_ha_time_s_chips = []
_phase_deg_uw_chips = []

for ha_c, dc_c in zip(ha_chips, corr_dc_chips):
    _ha_time_s_chips.append(ha_c / _omega_sid_deg_s)
    _phase_deg_uw_chips.append(np.degrees(np.unwrap(np.angle(dc_c[:, K_CH]))))

fig, ax = plot_unwrapped_phase_vs_ha_time(
    ha_time_s_chips=_ha_time_s_chips,
    phase_deg_chips=_phase_deg_uw_chips,
    sidereal_rate_deg_s=_omega_sid_deg_s,
    channel_index=K_CH,
    channel_freq_ghz=F_SKY_GHZ[K_CH],
    ha_time_limits_s=(ha_deg.min() / _omega_sid_deg_s, ha_deg.max() / _omega_sid_deg_s),
)
plt.show()
